In [5]:
import pandas as pd

# Definir la ruta del archivo
## Ya que el archivo esta en la misma carpeta que el script solo necesitamos poner su nombre

ruta_archivo = "products_asos.csv"

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

## Since were using the price column a lot we want to clean it first
df = pd.read_csv('products_asos.csv', on_bad_lines='skip')
df ['price'] = pd.to_numeric(df['price'], errors='coerce')  ## To convert everything to a number and set the rest to NaN
df = df.dropna(subset=['price']) ## Will dropp all the rows that have NaN in the price column

print(f"Data Loaded: {len(df)}rows")
df.head()

Data Loaded: 18378rows


,url,name,size,category,price,color,sku,description,images
0,https://www.asos.com/stradivarius/stradivarius...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...
1,https://www.asos.com/stradivarius/stradivarius...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...
2,https://www.asos.com/asos-design/asos-design-l...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...
3,https://www.asos.com/new-look/new-look-trench-...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...
4,https://www.asos.com/stradivarius/stradivarius...,Stradivarius double breasted wool coat in grey,"XS - UK 6,S - UK 8,M - UK 10,L - UK 12,XL - UK 14",Stradivarius double breasted wool coat in grey,59.99,GREY,123650194.0,[{'Product Details': 'Coats & Jackets by Strad...,['https://images.asos-media.com/products/strad...


In [10]:
df['description'] = df['description'].astype(str)

def get_brand (text):
    if 'by' in text:
      try:
        return text.split('by ')[1].split(' ')[0]
      except:
        return "Unknown"
        return "Unknown" 
      
df['brand_raw'] = df['description'].apply(get_brand)

In [11]:
df.head(3)

,url,name,size,category,price,color,sku,description,images,brand_raw
0,https://www.asos.com/stradivarius/stradivarius...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...,New
1,https://www.asos.com/stradivarius/stradivarius...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...,New
2,https://www.asos.com/asos-design/asos-design-l...,New Look trench coat in camel,"UK 4,UK 6,UK 8,UK 10,UK 12,UK 14 - Out of stoc...",New Look trench coat in camel,49.99,Neutral,126704571.0,[{'Product Details': 'Coats & Jackets by New L...,['https://images.asos-media.com/products/new-l...,New


In [12]:
brand_map = {
    'New': 'New Look',
    'River': 'River Island',
    'Miss': 'Miss Selfridge',
    'TopshopWelcome': 'Topshop'
}

df['brand'] = df['brand_raw'].map(brand_map).fillna(df['brand_raw'])

brand_counts = df['brand'].value_counts()
valid_brands = brand_counts[brand_counts >= 5].index
df_clean = df[df['brand'].isin(valid_brands).copy()]

print(df_clean['brand'].value_counts().head(10))

brand
ASOS                  4844
Topshop               1017
New Look               511
River Island           467
Miss Selfridge         429
adidas                 384
Vero                   327
The                    303
CollusionExclusive     282
&                      224
Name: count, dtype: int64


In [18]:
# 1.- Calculate to analyze stockouts
def calculate_phantom_revenue(size_str):
    if not isinstance(size_str, str):
        return 0, 0.0
    # Split "UK 6, UK 8 -Out of Stock" into list
    sizes = size_str.split(',')
    total_sizes = len(sizes)

    # Count how many items are out of stock
    out_of_stock_count = size_str.count('Out of Stock')

    # Calculate rate (0.0 to 1.0)
    rate = out_of_stock_count / total_sizes if total_sizes > 0 else 0.0
    return out_of_stock_count, rate

metrics = df_clean['size'].apply(lambda x: calculate_phantom_revenue(x))

df_clean['Stockout_Count'] = [x[0] for x in metrics]
df_clean['Stockout_Rate'] = [x[1] for x in metrics]

df_clean['Lost_Revenue'] = df_clean['price'] * df_clean['Stockout_Count']

cols = ['brand', 'name', 'price', 'Stockout_Count', 'Lost_Revenue']
print(df_clean.sort_values(by='Lost_Revenue', ascending=False).head(10)[cols])

              brand                                               name  price  \
0          New Look                      New Look trench coat in camel  49.99   
20410          ASOS  ASOS DESIGN mesh bodysuit with bra detail in b...  22.00   
20377      Carhartt  Carhartt WIP relaxed overdye long sleeve t-shi...  55.00   
20378          ASOS         ASOS DESIGN sequin wrap bodysuit in silver  36.00   
20379         Tammy  Tammy Girl Plus baby t-shirt with layered appl...  18.00   
20380          ASOS  ASOS DESIGN Maternity textured oversized t-shi...  16.00   
20382      Carhartt      Carhartt WIP locker oversize t-shirt in white  50.00   
20384  River Island  River Island embriodered ring detail crop top ...  33.00   
20386          ASOS  ASOS DESIGN Curve lace up front long sleeve bo...  25.00   
20389          ASOS  ASOS DESIGN Maternity ultimate slim fit t-shir...  14.00   

       Stockout_Count  Lost_Revenue  
0                   0           0.0  
20410               0           

/var/folders/s6/51m3ly7x3nxclqhc7891fthc0000gn/T/ipykernel_1020/3536440983.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Stockout_Count'] = [x[0] for x in metrics]
/var/folders/s6/51m3ly7x3nxclqhc7891fthc0000gn/T/ipykernel_1020/3536440983.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Stockout_Rate'] = [x[1] for x in metrics]
/var/folders/s6/51m3ly7x3nxclqhc7891fthc0000gn/T/ipykernel_1020/3536440983.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a s